<a href="https://colab.research.google.com/github/Tensor-Reloaded/IOAI-Workshop-CV-1/blob/main/vit/5.vit-ensemble-and-soup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multitask - the OxfordIIITPet dataset for segmentation and classification

In [28]:
import torch
import torch.nn as nn
from torchvision.transforms import v2
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader, Dataset
import timm
import torchvision
from torchvision.transforms.v2.functional import hflip
from tqdm import tqdm
import matplotlib.pyplot as plt

In [29]:
img_size = 224
num_classes = 37

In [30]:
class ClassificationDataset(Dataset):
    def __init__(self, split, image_transforms):
        self.data = OxfordIIITPet(
            root="../data",
            download=True,
            split=split,
            target_types=("category",),
        )
        self.image_transforms = image_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        image, class_label = self.data[i]
        image = self.image_transforms(image)
        return image, class_label


image_transforms_train = v2.Compose([
    v2.ToImage(),
    v2.Resize([img_size, img_size]),
    v2.RandomCrop([img_size, img_size], padding=12),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.AutoAugment(),
])

image_transforms_test = v2.Compose([
    v2.ToImage(),
    v2.Resize([img_size, img_size]),
    v2.ToDtype(torch.float32, scale=True),
])

cutmix_or_mixup = v2.RandomChoice([
    v2.CutMix(num_classes=num_classes),
    v2.MixUp(num_classes=num_classes),
])

train_dataset = ClassificationDataset("trainval", image_transforms_train)
test_dataset = ClassificationDataset("test", image_transforms_test)

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=32, drop_last=True)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=32, drop_last=False)

In [31]:
class ClassificationModel(nn.Module):
    def __init__(self, backbone_name='resnet18', num_classes=num_classes):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True)
        if not hasattr(self.backbone, "fc") and not hasattr(self.backbone, "head"):
            raise RuntimeError("Backbone not implemented: " + backbone_name)
        if not hasattr(self.backbone, "fc"):  # Has a head
            if hasattr(self.backbone.head, "fc"): # Is maxvit
                self.backbone.head.fc = nn.Linear(self.backbone.head.fc.weight.size(1), num_classes)
            else:
                # Is ViT
                self.backbone.head = nn.Linear(self.backbone.head.weight.size(1), num_classes)
        else:
            # Is resnet-like
            self.backbone.fc = nn.Linear(self.backbone.fc.weight.size(1), num_classes)

    def forward(self, x):
        return self.backbone(x)

    def freeze_backbone(self):
        self.backbone.requires_grad_(False)
        if not hasattr(self.backbone, "fc"):
            self.backbone.head.requires_grad_(True)
        else:
            self.backbone.fc.requires_grad_(True)

In [32]:
def f1_score(x, y):
    x_sum = x.sum().item()
    y_sum = y.sum().item()

    if x_sum == y_sum == 0:
        return 1.0
    elif x_sum == 0 or y_sum == 0:
        return 0.0

    return 2.0 * (x & y).sum().item() / (x_sum + y_sum)


def f1_macro(predicted, targets, num_classes):
    f1s = []
    for cls in range(num_classes):
        f1s.append(f1_score(predicted == cls, targets == cls))
    return sum(f1s) / num_classes


In [33]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
print("Using device", device)
model = ClassificationModel().to(device)
classification_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

Using device cuda


In [34]:
def train():
    loss_sum = 0.0
    num_batches = 0

    model.train()
    pbar = tqdm(train_loader, desc="Training")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        images, labels = cutmix_or_mixup(images, labels)

        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            class_logits = model(images)
            loss = classification_criterion(class_logits, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_sum += loss.item()
        num_batches += 1

        pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
        })

    return loss_sum / num_batches


@torch.inference_mode()
def val():
    cls_f1_sum = 0.0
    cls_acc_sum = 0.0
    loss_sum = 0.0
    num_batches = 0

    model.eval()
    pbar = tqdm(test_loader, desc="Evaluating")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            class_logits = model(images)
            loss = classification_criterion(class_logits, labels)

            pred_classes = class_logits.argmax(dim=1)

            cls_f1 = f1_macro(pred_classes, labels, num_classes=num_classes)
            cls_acc = (pred_classes == labels).float().mean().item()

        cls_f1_sum += cls_f1
        cls_acc_sum += cls_acc
        loss_sum += loss
        num_batches += 1

        pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'ClsF1': f'{cls_f1:.4f}',
            'ClsAcc': f'{cls_acc:.4f}',
        })

    return (
        cls_f1_sum / num_batches,
        cls_acc_sum / num_batches,
        loss_sum / num_batches,
    )


@torch.inference_mode()
def val_tta(tta_type):
    cls_f1_sum = 0.0
    cls_acc_sum = 0.0
    num_batches = 0

    model.eval()
    pbar = tqdm(test_loader, desc=f"Evaluating with TTA level {level}")

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            combined = [images]
            if tta_type == "mirroring":
                combined.append(hflip(images))
            elif tta_type == "translate":  # left
                padding_size = 2
                padded = v2.functional.pad(images, [padding_size])
                for i in [-2, 0, 2]:
                    for j in [-2, 0, 2]:
                        if i == 0 and j == 0:
                            continue
                        x = padding_size + i
                        y = padding_size + j
                        combined.append(padded[:, :, x:x + img_size, y:y + img_size])
            elif tta_type == "mirroring_and_translate":
                combined.append(hflip(images))
                padding_size = 2
                padded = v2.functional.pad(images, [padding_size])
                for i in [-2, 0, 2]:
                    for j in [-2, 0, 2]:
                        if i == 0 and j == 0:
                            continue
                        x = padding_size + i
                        y = padding_size + j
                        aux = padded[:, :, x:x + img_size, y:y + img_size]
                        combined.append(aux)
                        combined.append(hflip(aux))
            elif tta_type == "translate_aggressive":
                padding_size = 4
                padded = v2.functional.pad(images, [padding_size])
                for i in [-4, -2, 0, 2, 4]:
                    for j in [-4, -2, 0, 2, 4]:
                        if i == 0 and j == 0:
                            continue
                        x = padding_size + i
                        y = padding_size + j
                        combined.append(padded[:, :, x:x + img_size, y:y + img_size])

            outputs = sum(model(x) for x in combined)
        outputs = outputs.argmax(dim=1)

        cls_f1 = f1_macro(outputs, labels, num_classes=num_classes)
        cls_acc = (outputs == labels).float().mean().item()

        cls_f1_sum += cls_f1
        cls_acc_sum += cls_acc
        num_batches += 1

        pbar.set_postfix({
            'ClsF1': f'{cls_f1:.4f}',
            'ClsAcc': f'{cls_acc:.4f}',
        })

    return (
        cls_f1_sum / num_batches,
        cls_acc_sum / num_batches,
    )


@torch.inference_mode()
def val_tta_ensemble(ensemble_models, tta_type):
    cls_f1_sum = 0.0
    cls_acc_sum = 0.0
    num_batches = 0

    pbar = tqdm(test_loader, desc=f"Evaluating ENSEMBLE with TTA level {tta_type}")

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            combined = [images]
            if tta_type == "mirroring":
                combined.append(hflip(images))
            elif tta_type == "translate":  # left
                padding_size = 2
                padded = v2.functional.pad(images, [padding_size])
                for i in [-2, 0, 2]:
                    for j in [-2, 0, 2]:
                        if i == 0 and j == 0:
                            continue
                        x = padding_size + i
                        y = padding_size + j
                        combined.append(padded[:, :, x:x + img_size, y:y + img_size])
            elif tta_type == "mirroring_and_translate":
                combined.append(hflip(images))
                padding_size = 2
                padded = v2.functional.pad(images, [padding_size])
                for i in [-2, 0, 2]:
                    for j in [-2, 0, 2]:
                        if i == 0 and j == 0:
                            continue
                        x = padding_size + i
                        y = padding_size + j
                        aux = padded[:, :, x:x + img_size, y:y + img_size]
                        combined.append(aux)
                        combined.append(hflip(aux))
            elif tta_type == "translate_aggressive":
                padding_size = 4
                padded = v2.functional.pad(images, [padding_size])
                for i in [-4, -2, 0, 2, 4]:
                    for j in [-4, -2, 0, 2, 4]:
                        if i == 0 and j == 0:
                            continue
                        x = padding_size + i
                        y = padding_size + j
                        combined.append(padded[:, :, x:x + img_size, y:y + img_size])

            outputs = None
            for model in ensemble_models:
                model_outputs = sum(model(x) for x in combined) / len(combined)
                outputs = model_outputs if outputs is None else outputs + model_outputs
            outputs /= len(ensemble_models)

        outputs = outputs.argmax(dim=1)

        cls_f1 = f1_macro(outputs, labels, num_classes=num_classes)
        cls_acc = (outputs == labels).float().mean().item()

        cls_f1_sum += cls_f1
        cls_acc_sum += cls_acc
        num_batches += 1

        pbar.set_postfix({
            'ClsF1': f'{cls_f1:.4f}',
            'ClsAcc': f'{cls_acc:.4f}',
        })

    return (
        cls_f1_sum / num_batches,
        cls_acc_sum / num_batches,
    )


In [49]:
model_name = "hf_hub:timm/maxvit_tiny_tf_224.in1k"
# Loading all models first
model = ClassificationModel(model_name)

In [39]:
ensemble_size = 3
num_epochs = 5

for i in range(ensemble_size):
    model = ClassificationModel(model_name).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    model.freeze_backbone()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        tr_loss = train()
        vl_cls_f1, vl_cls_acc, vl_loss = val()
        print(f"[Train]                                | Loss: {tr_loss:.4f}")
        print(f"[Val]   ClsF1: {vl_cls_f1:.4f} | ClsAcc: {vl_cls_acc:.4f} | Loss: {vl_loss:.4f}")

    torch.save(model.state_dict(), f"{model_name.split('/')[-1]}_ensemble_{i}.pt")




Epoch 1/5


Evaluating: 100%|██████████| 115/115 [00:27<00:00,  4.21it/s, Loss=0.1856, ClsF1=1.0000, ClsAcc=1.0000]


[Train]                                | Loss: 2.0677
[Val]   ClsF1: 0.9562 | ClsAcc: 0.8649 | Loss: 0.5482

Epoch 2/5


Evaluating: 100%|██████████| 115/115 [00:28<00:00,  4.07it/s, Loss=0.1700, ClsF1=1.0000, ClsAcc=1.0000]


[Train]                                | Loss: 1.6391
[Val]   ClsF1: 0.9609 | ClsAcc: 0.9000 | Loss: 0.4773

Epoch 3/5


...

[Train]                                | Loss: 1.4917
[Val]   ClsF1: 0.9612 | ClsAcc: 0.8981 | Loss: 0.4206

Epoch 4/5


Evaluating: 100%|██████████| 115/115 [00:27<00:00,  4.19it/s, Loss=0.2185, ClsF1=1.0000, ClsAcc=1.0000]


[Train]                                | Loss: 1.5407
[Val]   ClsF1: 0.9627 | ClsAcc: 0.8954 | Loss: 0.4352

Epoch 5/5


Evaluating: 100%|██████████| 115/115 [00:27<00:00,  4.20it/s, Loss=0.2188, ClsF1=1.0000, ClsAcc=1.0000]


[Train]                                | Loss: 1.4532
[Val]   ClsF1: 0.9647 | ClsAcc: 0.9022 | Loss: 0.4363


In [52]:
results = []

In [53]:
tta_types = ["no_tta", "mirroring", "translate"]

In [54]:
ensemble_models = []
for i in range(ensemble_size):
    model = ClassificationModel(model_name).to(device)
    model.load_state_dict(torch.load(f"{model_name.split('/')[-1]}_ensemble_{i}.pt"))
    model.eval()
    for level in tta_types:
        val_f1_tta, val_acc_tta = val_tta(level)
        results.append((f"{model_name}_{i}", level, val_f1_tta, val_acc_tta))
    ensemble_models.append(model)


for level in ["no_tta", "mirroring", "translate"]:
    val_f1_tta, val_acc_tta = val_tta_ensemble(ensemble_models, level)
    results.append((f"{model_name}_ensemble", level, val_f1_tta, val_acc_tta))


Evaluating ENSEMBLE with TTA level translate: 100%|██████████| 115/115 [05:56<00:00,  3.10s/it, ClsF1=1.0000, ClsAcc=1.0000]


In [55]:
def load_state_dicts(model_name, n=5):
    state_dicts = []
    for i in range(n):
        path = f"{model_name.split('/')[-1]}_ensemble_{i}.pt"
        state_dicts.append(torch.load(path, map_location='cpu'))
    return state_dicts


def average_state_dicts(state_dicts):
    avg_state_dict = {}
    for key in state_dicts[0].keys():
        avg_state_dict[key] = sum(d[key] for d in state_dicts) / len(state_dicts)
    return avg_state_dict


def create_model_soup(model_name, n=5):
    state_dicts = load_state_dicts(model_name, n=n)
    avg_state = average_state_dicts(state_dicts)

    model = ClassificationModel(model_name).to(device)
    model.load_state_dict(avg_state)
    return model



In [56]:
soup_model = create_model_soup(model_name, n=ensemble_size)
soup_model.eval()

for level in tta_types:
    model = soup_model
    val_f1_tta, val_acc_tta = val_tta(level)
    results.append((f"{model_name}_soup", level, val_f1_tta, val_acc_tta))


Evaluating with TTA level translate: 100%|██████████| 115/115 [02:09<00:00,  1.13s/it, ClsF1=1.0000, ClsAcc=1.0000]


In [57]:
for name, level, val_f1_tta, val_acc_tta in results:
    print(f"{name.split('/')[-1]: <35} | {level: <10} | ClsF1: {val_f1_tta:.4f} | ClsAcc: {val_acc_tta:.4f}")

maxvit_tiny_tf_224.in1k_0           | no_tta     | ClsF1: 0.9583 | ClsAcc: 0.8984
maxvit_tiny_tf_224.in1k_0           | mirroring  | ClsF1: 0.9603 | ClsAcc: 0.9019
maxvit_tiny_tf_224.in1k_0           | translate  | ClsF1: 0.9617 | ClsAcc: 0.9033
maxvit_tiny_tf_224.in1k_1           | no_tta     | ClsF1: 0.9622 | ClsAcc: 0.9027
maxvit_tiny_tf_224.in1k_1           | mirroring  | ClsF1: 0.9624 | ClsAcc: 0.9016
maxvit_tiny_tf_224.in1k_1           | translate  | ClsF1: 0.9658 | ClsAcc: 0.9079
maxvit_tiny_tf_224.in1k_2           | no_tta     | ClsF1: 0.9647 | ClsAcc: 0.9022
maxvit_tiny_tf_224.in1k_2           | mirroring  | ClsF1: 0.9656 | ClsAcc: 0.9054
maxvit_tiny_tf_224.in1k_2           | translate  | ClsF1: 0.9677 | ClsAcc: 0.9060
maxvit_tiny_tf_224.in1k_ensemble    | no_tta     | ClsF1: 0.9627 | ClsAcc: 0.9057
maxvit_tiny_tf_224.in1k_ensemble    | mirroring  | ClsF1: 0.9632 | ClsAcc: 0.9073
maxvit_tiny_tf_224.in1k_ensemble    | translate  | ClsF1: 0.9671 | ClsAcc: 0.9141
maxvit_tiny_tf_2

In [58]:
for name, level, val_f1_tta, val_acc_tta in results[:9]:
    print(f"{name.split('/')[-1]: <35} | {level: <10} | ClsF1: {val_f1_tta:.4f} | ClsAcc: {val_acc_tta:.4f}")

maxvit_tiny_tf_224.in1k_0           | no_tta     | ClsF1: 0.9583 | ClsAcc: 0.8984
maxvit_tiny_tf_224.in1k_0           | mirroring  | ClsF1: 0.9603 | ClsAcc: 0.9019
maxvit_tiny_tf_224.in1k_0           | translate  | ClsF1: 0.9617 | ClsAcc: 0.9033
maxvit_tiny_tf_224.in1k_1           | no_tta     | ClsF1: 0.9622 | ClsAcc: 0.9027
maxvit_tiny_tf_224.in1k_1           | mirroring  | ClsF1: 0.9624 | ClsAcc: 0.9016
maxvit_tiny_tf_224.in1k_1           | translate  | ClsF1: 0.9658 | ClsAcc: 0.9079
maxvit_tiny_tf_224.in1k_2           | no_tta     | ClsF1: 0.9647 | ClsAcc: 0.9022
maxvit_tiny_tf_224.in1k_2           | mirroring  | ClsF1: 0.9656 | ClsAcc: 0.9054
maxvit_tiny_tf_224.in1k_2           | translate  | ClsF1: 0.9677 | ClsAcc: 0.9060


In [59]:
for name, level, val_f1_tta, val_acc_tta in results[9:]:
    print(f"{name.split('/')[-1]: <35} | {level: <10} | ClsF1: {val_f1_tta:.4f} | ClsAcc: {val_acc_tta:.4f}")

maxvit_tiny_tf_224.in1k_ensemble    | no_tta     | ClsF1: 0.9627 | ClsAcc: 0.9057
maxvit_tiny_tf_224.in1k_ensemble    | mirroring  | ClsF1: 0.9632 | ClsAcc: 0.9073
maxvit_tiny_tf_224.in1k_ensemble    | translate  | ClsF1: 0.9671 | ClsAcc: 0.9141
maxvit_tiny_tf_224.in1k_soup        | no_tta     | ClsF1: 0.9642 | ClsAcc: 0.9087
maxvit_tiny_tf_224.in1k_soup        | mirroring  | ClsF1: 0.9632 | ClsAcc: 0.9079
maxvit_tiny_tf_224.in1k_soup        | translate  | ClsF1: 0.9669 | ClsAcc: 0.9144
